# Лабораторная 13. Перевод с русского на английский с собственной GRU
Постройте переводчик коротких предложений. Энкодер читает русские слова, декодер по одному выдаёт английские. Обе сети используют реализованную вами GRU (gated recurrent unit). У них разные обучаемые веса. Готовые nn.GRU и nn.GRUCell использовать нельзя.

Пять задач по 2 балла. После реализации запустите обучение с нуля, сохраните в сдаваемом ноутбуке результаты на отложенных парах и разберите два перевода. Высокий балл за реализацию не требует угадать конкретную формулировку эталонного перевода.

In [ ]:
import csv
from pathlib import Path
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.data import DataLoader
from torch.nn.utils.rnn import pad_sequence

torch.set_num_threads(2)
torch.manual_seed(13)
PAD, BOS, EOS, UNK = range(4)

## Параллельный корпус
В выборке 3200 пар длиной от двух до шести слов на каждом языке, с частотной лексикой. Фразы приведены к нижнему регистру, конечные знаки препинания удалены, повторяющиеся русские или английские предложения исключены. Пары уже перемешаны с фиксированным seed. Первые 2800 используйте для обучения, следующие 200 для валидации, последние 200 для итоговой оценки. Словари строим только по обучению. Для каждого предложения в TSV сохранены имена авторов и ID оригиналов. [Tatoeba через ManyThings](https://www.manythings.org/anki/) · [CC BY 2.0 FR](https://creativecommons.org/licenses/by/2.0/fr/)

In [ ]:
filename = Path("data/translation_ru_en.tsv")
if not filename.is_file():
    filename = Path("../ml_dafe_mipt/classes/week_13_text_models") / filename
with filename.open(encoding="utf-8") as f:
    pairs = [(row["ru"], row["en"]) for row in csv.DictReader(f, delimiter="	")]
train, valid, test = pairs[:2800], pairs[2800:3000], pairs[3000:]
def vocabulary(texts):
    words = sorted({word for text in texts for word in text.split()})
    return ["<pad>", "<bos>", "<eos>", "<unk>", *words]
ru = vocabulary(row[0] for row in train)
en = vocabulary(row[1] for row in train)
ru_id = {w:i for i,w in enumerate(ru)}
en_id = {w:i for i,w in enumerate(en)}
for source, target in train[:4]:
    print(source, "→", target)

In [ ]:
def ids(words, vocab):
    return [vocab.get(w, UNK) for w in words.split()]

def collate(pairs):
    sources = [torch.tensor(ids(s, ru_id) + [EOS]) for s, _ in pairs]
    targets = [torch.tensor([BOS] + ids(t, en_id) + [EOS]) for _, t in pairs]
    return tuple(pad_sequence(items, batch_first=True, padding_value=PAD)
                 for items in [sources, targets])

train_loader = DataLoader(train, batch_size=48, shuffle=True, collate_fn=collate,
                          generator=torch.Generator().manual_seed(17))
source, target = collate(train[:3])
print("Источник:", source, "\nПеревод с BOS и EOS:", target)

## 1. Шаг GRU · 2 балла
У GRU одно состояние $h$. Получите две проекции: $a=W_x x+b_x$ и $b=W_h h+b_h$. Каждая содержит три блока ширины $H$ в порядке reset, update, candidate.
$$r=\sigma(a_r+b_r),\quad z=\sigma(a_z+b_z),\quad g=\tanh(a_g+r\odot b_g),\quad h'=z\odot h+(1-z)\odot g.$$
В соглашении PyTorch $z$ задаёт долю прежнего состояния. $r$ регулирует, сколько прошлого попадёт в кандидат. Верните новое состояние с поддержкой autograd.

Сравните с LSTM. Там входной вентиль и вентиль забывания выбираются независимо и есть отдельная память c. Здесь у записи и переноса коэффициенты 1−z и z: при z, близком к единице, новое состояние почти равно прежнему. При малом r прошлое слабо влияет на кандидат, но всё ещё может переноситься напрямую через z.

In [ ]:
# x: (B,D), h: (B,H), wx: D -> 3H, wh: H -> 3H; результат (B,H)
def gru_step(x: torch.Tensor, h: torch.Tensor, wx: nn.Linear, wh: nn.Linear) -> torch.Tensor:
    ...

In [ ]:
# --- проверка ---
wx = nn.Linear(2, 9, bias=False)
wh = nn.Linear(3, 9, bias=False)
with torch.no_grad():
    wx.weight.zero_()
    wh.weight.zero_()
torch.testing.assert_close(gru_step(torch.zeros(2,2), torch.ones(2,3), wx, wh), torch.full((2,3), .5))

## Слои переводчика
Энкодер и декодер получают по эмбеддингу и по две линейные проекции. Веса каждой проекции общие для всех позиций соответствующего языка. Английская выходная голова переводит состояние декодера в логиты словаря.

In [ ]:
class TranslatorLayers(nn.Module):
    def __init__(self, n_ru, n_en, width=48, hidden=72):
        super().__init__()
        self.ru = nn.Embedding(n_ru, width, padding_idx=PAD)
        self.en = nn.Embedding(n_en, width, padding_idx=PAD)
        self.ex = nn.Linear(width, hidden*3)
        self.eh = nn.Linear(hidden, hidden*3)
        self.dx = nn.Linear(width, hidden*3)
        self.dh = nn.Linear(hidden, hidden*3)
        self.head = nn.Linear(hidden, n_en)
        self.hidden = hidden

## 2. Энкодер · 2 балла
Начните с нулевого состояния. Прочитайте каждый настоящий токен русского батча, включая EOS, своей функцией gru_step. Справа стоят PAD. Для каждой строки верните состояние после её последнего настоящего токена, даже если соседняя фраза длиннее.

In [ ]:
# source: индексы (B,S); результат (B,H)
def encode_ru(model: TranslatorLayers, source: torch.Tensor) -> torch.Tensor:
    ...

In [ ]:
# --- проверка ---
probe = TranslatorLayers(10, 12)
a = encode_ru(probe, torch.tensor([[4, EOS]]))
b = encode_ru(probe, torch.tensor([[4, EOS, PAD], [5, 6, EOS]]))
torch.testing.assert_close(a[0], b[0])

## 3. Декодер на обучении · 2 балла
Первое состояние приходит из энкодера. Для каждого входного английского слова сделайте один шаг своей GRU и примените голову. В previous передают BOS и правильные слова перевода до последней позиции. Верните логиты до softmax, включая позиции, которые позже исключит loss.

In [ ]:
# previous: (B,T), state: (B,H); результат (B,T,V_en)
def teacher_logits(model: TranslatorLayers, previous: torch.Tensor, state: torch.Tensor) -> torch.Tensor:
    ...

In [ ]:
# --- проверка ---
h = torch.randn(2, probe.hidden)
previous = torch.tensor([[BOS, 5], [BOS, 7]])
out = teacher_logits(probe, previous, h)
assert out.shape == (2, 2, 12)
torch.testing.assert_close(out[:, 0], teacher_logits(probe, previous[:, :1], h)[:, 0])

## Собираем переводчик
Теперь обе функции определены. Метод `forward` передаёт состояние русского энкодера английскому декодеру.

In [ ]:
class Translator(TranslatorLayers):
    def forward(self, source, previous):
        state = encode_ru(self, source)
        return teacher_logits(self, previous, state)

## 4. Обновление на батче · 2 балла
В target уже есть BOS, слова, EOS и правые PAD. Выполните одно обновление переданным оптимизатором. Используйте teacher forcing, усредните cross-entropy по настоящим целевым токенам, включая EOS. Перед backward очистите старые градиенты. Верните значение потерь до обновления как float.

In [ ]:
# source: (B,S), target: (B,T+1); результат: средние потери на символе словаря (токене)
def train_batch(model: Translator, optimizer: torch.optim.Optimizer,
                source: torch.Tensor, target: torch.Tensor) -> float:
    ...

In [ ]:
# --- проверка ---
src = torch.tensor([[4, EOS], [7, EOS]])
tgt = torch.tensor([[BOS, 5, EOS, PAD], [BOS, 6, 4, EOS]])
probe = Translator(10, 12)
before = probe.head.weight.detach().clone()
value = train_batch(probe, torch.optim.SGD(probe.parameters(), lr=.01), src, tgt)
assert value > 0 and not torch.equal(before, probe.head.weight)

## 5. Перевод без эталона · 2 балла
Закодируйте источник один раз. Декодер получает BOS, затем собственное предыдущее предсказание. Выбирайте argmax среди всех токенов кроме BOS и PAD. В каждой строке оставьте первый EOS, заполните хвост PAD. Остановитесь, когда все строки завершены или достигнут limit. При limit=0 верните пустое продолжение. Параметры модели при переводе сохраняются.

In [ ]:
# source: (B,S); результат: английские индексы (B,limit) без начального BOS
def greedy_en(model: Translator, source: torch.Tensor, limit: int) -> torch.Tensor:
    ...

In [ ]:
# --- проверка ---
with torch.no_grad():
    probe.head.weight.zero_()
    probe.head.bias.zero_()
    probe.head.bias[PAD] = 20
    probe.head.bias[EOS] = 10
assert greedy_en(probe, src, 3).tolist() == [[EOS,PAD,PAD], [EOS,PAD,PAD]]

## Обучение с нуля
Выберите эпоху по средней потере на валидации. На test проверьте выбранные веса один раз. Кривые с разным числом дополнений сравнимы, если суммировать потери и делить на число настоящих токенов.

In [ ]:
from copy import deepcopy

@torch.no_grad()
def evaluate_loss(model, rows):
    model.eval()
    total, count = 0., 0
    for src, tgt in DataLoader(rows, batch_size=64, collate_fn=collate):
        logits = model(src, tgt[:, :-1])
        labels = tgt[:, 1:]
        total += F.cross_entropy(logits.flatten(0,1), labels.flatten(), ignore_index=PAD, reduction="sum").item()
        count += int(labels.ne(PAD).sum())
    return total / count

torch.manual_seed(13)
model = Translator(len(ru), len(en))
optimizer = torch.optim.Adam(model.parameters(), lr=.004)
best = float("inf")
weights = None
for epoch in range(1, 13):
    total, count = 0., 0
    for src, tgt in train_loader:
        tokens = int(tgt[:, 1:].ne(PAD).sum())
        total += train_batch(model, optimizer, src, tgt) * tokens
        count += tokens
    score = evaluate_loss(model, valid)
    if score < best:
        best, weights = score, deepcopy(model.state_dict())
    print(f"{epoch:2d}  train {total/count:.3f}  validation {score:.3f}")
if weights is None:
    raise RuntimeError("Не удалось выбрать эпоху по валидационной потере")
model.load_state_dict(weights)
model.eval()

In [ ]:
def english(row):
    words = []
    for token in row.tolist():
        if token == EOS:
            break
        if token != PAD:
            words.append(en[token])
    return " ".join(words)

sources, _ = collate(test)
answers = [english(row) for row in greedy_en(model, sources, limit=10)]
hits = sum(answer == reference for answer, (_, reference) in zip(answers, test))
print(f"Точное совпадение: {hits}/{len(test)}. Потеря на токен: {evaluate_loss(model,test):.3f}")
for (source, reference), answer in list(zip(test, answers))[:8]:
    print(f"{source}  →  {answer}   | эталон: {reference}")

Выберите два расхождения. Где модель действительно поменяла смысл? Где её вариант может быть допустимым переводом? Объясните различие между точным совпадением целой строки и потерей на токен при teacher forcing.